# Прекод

# Сборный проект-4

Вам поручено разработать демонстрационную версию поиска изображений по запросу.

Для демонстрационной версии нужно обучить модель, которая получит векторное представление изображения, векторное представление текста, а на выходе выдаст число от 0 до 1 — покажет, насколько текст и картинка подходят друг другу.

**Описание данных**

Данные доступны по [ссылке](https://code.s3.yandex.net/datasets/dsplus_integrated_project_4.zip).

В файле `train_dataset.csv` находится информация, необходимая для обучения: имя файла изображения, идентификатор описания и текст описания. Для одной картинки может быть доступно до 5 описаний. Идентификатор описания имеет формат `<имя файла изображения>#<порядковый номер описания>`.

В папке `train_images` содержатся изображения для тренировки модели.

В файле `CrowdAnnotations.tsv` — данные по соответствию изображения и описания, полученные с помощью краудсорсинга. Номера колонок и соответствующий тип данных:

1. Имя файла изображения.
2. Идентификатор описания.
3. Доля людей, подтвердивших, что описание соответствует изображению.
4. Количество человек, подтвердивших, что описание соответствует изображению.
5. Количество человек, подтвердивших, что описание не соответствует изображению.

В файле `ExpertAnnotations.tsv` содержатся данные по соответствию изображения и описания, полученные в результате опроса экспертов. Номера колонок и соответствующий тип данных:

1. Имя файла изображения.
2. Идентификатор описания.

3, 4, 5 — оценки трёх экспертов.

Эксперты ставят оценки по шкале от 1 до 4, где 1 — изображение и запрос совершенно не соответствуют друг другу, 2 — запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует, 3 — запрос и текст соответствуют с точностью до некоторых деталей, 4 — запрос и текст соответствуют полностью.

В файле `test_queries.csv` находится информация, необходимая для тестирования: идентификатор запроса, текст запроса и релевантное изображение. Для одной картинки может быть доступно до 5 описаний. Идентификатор описания имеет формат `<имя файла изображения>#<порядковый номер описания>`.

В папке `test_images` содержатся изображения для тестирования модели.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import transformers
import torch
from tqdm import tqdm
import os

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

import nltk
import re
from nltk.corpus import stopwords

import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Добавляем путь в NLTK
nltk_data_dir = '/kaggle/working/nltk_data'
nltk.data.path.append(nltk_data_dir)

from PIL import Image
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
transformers.logging.set_verbosity_error()
from transformers import BertTokenizer, BertModel, AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer

!pip install --upgrade keras-nlp keras
import keras_nlp


from tqdm import notebook

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

import sentence_transformers
from sentence_transformers import SentenceTransformer, util
import glob

import warnings
warnings.filterwarnings('ignore')

path = 'dsplus_integrated_project_4/to_upload'

## Исследовательский анализ данных

Наш датасет содержит экспертные и краудсорсинговые оценки соответствия текста и изображения.

В файле с экспертными мнениями для каждой пары изображение-текст имеются оценки от трёх специалистов. Для решения задачи вы должны эти оценки агрегировать — превратить в одну. Существует несколько способов агрегации оценок, самый простой — голосование большинства: за какую оценку проголосовала большая часть экспертов (в нашем случае 2 или 3), та оценка и ставится как итоговая. Поскольку число экспертов меньше числа классов, может случиться, что каждый эксперт поставит разные оценки, например: 1, 4, 2. В таком случае данную пару изображение-текст можно исключить из датасета.

Вы можете воспользоваться другим методом агрегации оценок или придумать свой.

В файле с краудсорсинговыми оценками информация расположена в таком порядке:

1. Доля исполнителей, подтвердивших, что текст **соответствует** картинке.
2. Количество исполнителей, подтвердивших, что текст **соответствует** картинке.
3. Количество исполнителей, подтвердивших, что текст **не соответствует** картинке.

После анализа экспертных и краудсорсинговых оценок выберите либо одну из них, либо объедините их в одну по какому-то критерию: например, оценка эксперта принимается с коэффициентом 0.6, а крауда — с коэффициентом 0.4.

Ваша модель должна возвращать на выходе вероятность соответствия изображения тексту, поэтому целевая переменная должна иметь значения от 0 до 1.


### Загрузка данных

In [ ]:
df_train = pd.read_csv(os.path.join(path, 'train_dataset.csv'))

df_crowd = pd.read_csv(os.path.join(path, 'CrowdAnnotations.tsv'),
                       sep='\t',
                       names=['image', 'query_id', 'share_pos', 'count_pos', 'count_neg'])

df_expert = pd.read_csv(os.path.join(path, 'ExpertAnnotations.tsv'),
                        sep='\t',
                        names=['image', 'query_id', 'first', 'second', 'third'])

df_queries = pd.read_csv(os.path.join(path, 'test_queries.csv'),
                         index_col=[0],
                         sep='|')

df_images = pd.read_csv(os.path.join(path, 'test_images.csv'),
                        sep='|')

In [ ]:
def describe_db(name):
    print(name.name)
    print(name.info())
    print(name.shape)
    print('Дубликаты:', name.duplicated().sum())
    print()
    print('Пропуски:')
    print(name.isna().sum())
    print()

df_train.name = "Train Dataset"
df_crowd.name = "Crowd Annotations"
df_expert.name = "Expert Annotations"
df_queries.name = "Test Queries"
df_images.name = "Images Info"

In [ ]:
data = [df_train, df_crowd, df_expert, df_queries, df_images]
for db in data:
    describe_db(db)
    display(db.head(20))

In [ ]:
print('Количество уникальных фото на трейне:', len(df_train['image'].unique()))
print('Количество уникальных фото на тесте:', len(df_queries['image'].unique()))
print('Количество уникальных запросов на тесте:', df_queries.drop_duplicates().shape[0])
print('Количество уникальных сочетаний фото-текст, оцененных экспертами:', df_expert.drop_duplicates().shape[0])
print('Количество уникальных сочетаний фото-текст, оцененных людьми:', df_crowd.drop_duplicates().shape[0])

Дубликатов и пропусков не наблюдается. Проверим так же на утечку данных (пересечение картинок в трейне и тесте). Пересечение текстов проверять не будем, так как текст с достаточно большой вероятностью может продублироваться и в реальных кейсах.

In [ ]:
print('Пересекающиеся картинки в тесте и трейне:', len(set(df_train['image']) & set(df_queries['image'])))

Картинки не пересекаются, утечки нет. Приступим к анализу данных.

### Анализ данных
#### Оценки экспертов

In [ ]:
def categ_pieplot(df, columns, title):
    all_values = pd.concat([df[col] for col in columns])

    value_counts = all_values.value_counts().sort_index()

    print("Распределение всех оценок (все эксперты):")
    display(value_counts)

    # Строим круговую диаграмму
    plt.figure(figsize=(8, 8))
    value_counts.plot.pie(autopct='%1.0f%%')
    plt.title(title)
    plt.ylabel('')
    plt.show()

categ_pieplot(df_expert, ['first', 'second', 'third'], 'Expert Annotations (All Experts)')

Судя по оценкам, чаще всего картинка совершенно не соответствует тексту (56%) по мнению экспертов. И только в 6% случаев эксперт ставил оценку 4. Проверим средние значения для картинок.

In [ ]:
def avg_score_barplot(df, columns, title):
    # Вычисляем точную среднюю
    df['avg_score'] = df[columns].mean(axis=1)

    # Округляем только для группировки
    rounded = df['avg_score'].round(2)
    value_counts = rounded.value_counts().sort_index()
    percentages = (value_counts / len(df) * 100).round(1)

    print("Распределение средних оценок:")
    display(value_counts)
    print(f"\nВсего изображений: {len(df)}")
    print(f"Средняя оценка: {df['avg_score'].mean():.3f}")

    plt.figure(figsize=(12, 8))
    bars = plt.bar(value_counts.index.astype(str), value_counts.values,
                   color='steelblue', edgecolor='black', alpha=0.7)
    plt.xlabel('Средняя оценка (по трём экспертам)', fontsize=12)
    plt.ylabel('Количество изображений', fontsize=12)
    plt.title(f'{title}\nРаспределение средних оценок', fontsize=14)

    for bar, count, pct in zip(bars, value_counts.values, percentages):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{count}\n({pct}%)', ha='center', va='bottom', fontsize=9)

    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return df

df_expert = avg_score_barplot(df_expert, ['first', 'second', 'third'], 'Expert Annotations')
df_expert.head()

Как видим, чаще всего (более 40%) средняя оценка для картинки - 1. То есть полное согласие о полном не соответствии. В 4.2% картинок эксперты полностью сходятся в оценке 4. Реже всего встречаются варианты, когда с оценкой полного соответствия не согласны 1 или 2 эксперта (и ставят оценку 3).

Такое распределение оценок говорит о том, что модель будет больше обучаться на отрицании сочетания картинок и текста. Есть сомнения в том, что получится высокая метрика. Однако, нельзя забывать, что для каждой картинки есть несколько описаний. Посмотрим оценки для картинки, а не описания. Посмотрим максимальное значение для средней оценки по описанию, чтоб понять, есть ли для картинок достаточно точное описание вообще. И посмотрим на среднее значение всех оценок для каждой картинки для понимания общей тенденции.

In [ ]:
def max_score_by_image_barplot(df, columns, title):

    image_max = df.groupby('image')['avg_score'].max().round(2)

    # Считаем частоту
    value_counts = image_max.value_counts().sort_index()
    percentages = (value_counts / len(image_max) * 100).round(1)

    print(f"Уникальных изображений: {len(image_max)}")
    print(f"\nРаспределение максимальных оценок по изображениям:")
    display(value_counts)

    # График
    plt.figure(figsize=(12, 8))
    bars = plt.bar(value_counts.index.astype(str), value_counts.values,
                   color='coral', edgecolor='black', alpha=0.7)
    plt.xlabel('Максимальная средняя оценка среди описаний изображения', fontsize=12)
    plt.ylabel('Количество уникальных изображений', fontsize=12)
    plt.title(f'{title}\nРаспределение максимальных оценок по изображениям', fontsize=14)

    for bar, count, pct in zip(bars, value_counts.values, percentages):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{count}\n({pct}%)', ha='center', va='bottom', fontsize=9)

    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return image_max

# Использование
image_max = max_score_by_image_barplot(df_expert, ['first', 'second', 'third'], 'Expert Annotations')

In [ ]:
def mean_score_by_image_histogram(df, columns, title):
    # Для каждого изображения берём СРЕДНЕЕ из средних оценок его описаний
    image_mean = df.groupby('image')['avg_score'].mean().round(2)

    print(f"Уникальных изображений: {len(image_mean)}")
    print(f"Средняя оценка по всем изображениям: {image_mean.mean():.2f}")
    print(f"Медиана: {image_mean.median():.2f}")
    print(f"Стандартное отклонение: {image_mean.std():.2f}")

    # Гистограмма
    plt.figure(figsize=(12, 6))
    plt.hist(image_mean, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    plt.xlabel('Средняя оценка изображения (усреднённая по всем описаниям)', fontsize=12)
    plt.ylabel('Количество уникальных изображений', fontsize=12)
    plt.title(f'{title}\nРаспределение средних оценок по изображениям (гистограмма)', fontsize=14)
    plt.axvline(image_mean.mean(), color='red', linestyle='--', linewidth=2, label=f'Среднее = {image_mean.mean():.2f}')
    plt.axvline(image_mean.median(), color='green', linestyle='--', linewidth=2, label=f'Медиана = {image_mean.median():.2f}')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return image_mean

# Использование
image_mean = mean_score_by_image_histogram(df_expert, ['first', 'second', 'third'], 'Expert Annotations')

Как мы видим, далеко не для каждой картинки нашлось идеальное описание (оценка 4 у 21% картинок). И всего у ~40% картинок есть описание со средней оценкой выше 3. Больше 22% изображений имеют наивысшую оценку менее 2. Значит, мы действительно имеет огромное количество данных, которое будет влияет на обучение "от обратного". Кроме того, формулировка оценки 2 ("запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует") достаточно расплывчата без точных указаний элементов и тоже будет довольно негативно влиять на результат.

По среднему средних оценок видна примерно та же тенденция - почти все значения стремятся к оценке ~1.3, что опять же говорит о сплошном отрицании сочетания текста и картинки.

Посмотрим на оценки краудсорсинга.

#### Оценки краудсорсинга

In [ ]:
def crowd_share_barplot(df, title):
    # Берём уникальные описания (по query_id)
    df_unique = df.drop_duplicates(subset=['query_id'])

    # Доля подтверждений для каждого уникального описания (округляем до 2 знаков)
    share_values = df_unique['share_pos'].round(2)

    # Считаем частоту каждого уникального значения
    value_counts = share_values.value_counts().sort_index()
    percentages = (value_counts / len(share_values) * 100).round(1)

    print(f"Уникальных описаний: {len(share_values)}")
    print(f"Средняя доля подтверждений: {share_values.mean():.2f}")
    print(f"\nРаспределение долей подтверждений:")
    display(value_counts)

    # Bar plot
    plt.figure(figsize=(14, 8))
    bars = plt.bar(value_counts.index.astype(str), value_counts.values,
                   color='steelblue', edgecolor='black', alpha=0.7)
    plt.xlabel('Доля людей, подтвердивших соответствие', fontsize=12)
    plt.ylabel('Количество уникальных описаний', fontsize=12)
    plt.title(f'{title}\nРаспределение доли подтверждений по уникальным описаниям', fontsize=14)

    for bar, count, pct in zip(bars, value_counts.values, percentages):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{count}\n({pct}%)', ha='center', va='bottom', fontsize=9)

    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Использование
crowd_share_barplot(df_crowd, 'Crowd Annotations')

In [ ]:
def crowd_mean_share_histogram(df, title):
    # Для каждого изображения берём СРЕДНЮЮ долю подтверждений по всем его описаниям
    image_mean_share = df.groupby('image')['share_pos'].mean().round(3)

    print(f"Уникальных изображений: {len(image_mean_share)}")
    print(f"Средняя доля подтверждений по всем изображениям: {image_mean_share.mean():.3f}")
    print(f"Медиана: {image_mean_share.median():.3f}")
    print(f"Стандартное отклонение: {image_mean_share.std():.3f}")
    print(f"Минимум: {image_mean_share.min():.3f}")
    print(f"Максимум: {image_mean_share.max():.3f}")

    # Гистограмма
    plt.figure(figsize=(12, 6))
    plt.hist(image_mean_share, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    plt.xlabel('Средняя доля людей, подтвердивших соответствие (по всем описаниям изображения)', fontsize=12)
    plt.ylabel('Количество уникальных изображений', fontsize=12)
    plt.title(f'{title}\nРаспределение средней доли подтверждений по изображениям', fontsize=14)
    plt.axvline(image_mean_share.mean(), color='red', linestyle='--', linewidth=2, label=f'Среднее = {image_mean_share.mean():.3f}')
    plt.axvline(image_mean_share.median(), color='green', linestyle='--', linewidth=2, label=f'Медиана = {image_mean_share.median():.3f}')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return image_mean_share

# Использование
crowd_mean_share = crowd_mean_share_histogram(df_crowd, 'Crowd Annotations')

In [ ]:
def crowd_pos_count_histogram(df, title):
    # Для каждого описания берём количество подтверждений
    # (можно grouped по изображению, но вы хотели гистограмму для людей подтвердивших описание)

    print(f"Всего описаний: {len(df)}")
    print(f"Общее количество подтверждений: {df['count_pos'].sum()}")
    print(f"Среднее количество подтверждений на описание: {df['count_pos'].mean():.2f}")
    print(f"Медиана: {df['count_pos'].median():.2f}")
    print(f"Максимум подтверждений у одного описания: {df['count_pos'].max()}")

    # Гистограмма распределения count_pos
    plt.figure(figsize=(12, 6))

    # Так как значения дискретные (0,1,2,3), лучше сделать bar plot
    value_counts = df['count_pos'].value_counts().sort_index()

    plt.bar(value_counts.index, value_counts.values, edgecolor='black', alpha=0.7, color='coral')
    plt.xlabel('Количество людей, подтвердивших соответствие', fontsize=12)
    plt.ylabel('Количество описаний', fontsize=12)
    plt.title(f'{title}\nРаспределение количества подтверждений на описание', fontsize=14)

    # Добавляем значения на столбцы
    for x, count in zip(value_counts.index, value_counts.values):
        plt.text(x, count + 5, str(count), ha='center', va='bottom', fontsize=9)

    plt.xticks([0, 1, 2, 3, 4, 5])
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    return value_counts

# Использование
crowd_pos_counts = crowd_pos_count_histogram(df_crowd, 'Crowd Annotations')

По оценкам из краудсорсинга складывается такая же картина - люди чаще всего не подтверждают соответствие текста и картинки. Посмотрим насколько эти оценки пересекаются с оценками от экспертов.

In [ ]:
def compare_crowd_expert_scatter(df_crowd, df_expert, title):
    # Для df_crowd: средняя доля подтверждений по изображению
    crowd_by_image = df_crowd.groupby('image')['share_pos'].mean().round(2)

    # Для df_expert: средняя оценка по изображению (усреднённая по описаниям)
    expert_by_image = df_expert.groupby('image')['avg_score'].mean().round(2)

    # Объединяем данные по image
    merged = pd.DataFrame({
        'crowd_share': crowd_by_image,
        'expert_score': expert_by_image
    }).dropna()

    print(f"Общих изображений: {len(merged)}")
    print(f"\nКорреляция Пирсона: {merged['crowd_share'].corr(merged['expert_score']):.3f}")
    print(f"Корреляция Спирмена: {merged['crowd_share'].corr(merged['expert_score'], method='spearman'):.3f}")

    # Scatter plot
    plt.figure(figsize=(10, 8))
    plt.scatter(merged['crowd_share'], merged['expert_score'], alpha=0.6, color='steelblue', edgecolor='black')
    plt.xlabel('Средняя доля подтверждений (краудсорсинг)', fontsize=12)
    plt.ylabel('Средняя оценка экспертов', fontsize=12)
    plt.title(f'{title}\nСравнение оценок краудсорсинга и экспертов по изображениям', fontsize=14)

    # Линия регрессии
    z = np.polyfit(merged['crowd_share'], merged['expert_score'], 1)
    p = np.poly1d(z)
    plt.plot(merged['crowd_share'].sort_values(), p(merged['crowd_share'].sort_values()),
             color='red', linestyle='--', linewidth=2, label=f'Линия регрессии (r={merged["crowd_share"].corr(merged["expert_score"]):.2f})')

    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return merged

# Использование
merged = compare_crowd_expert_scatter(df_crowd, df_expert, 'Crowd vs Expert')

- Краудсорсинговые данные (доля подтверждений) достаточно хорошо согласуются с экспертными оценками
- Разница между Пирсоном (0.50) и Спирменом (0.54) небольшая — связь близка к линейной

Объединим оценки и внесем их в основную таблицу как целевое значение

In [ ]:


df_scores = pd.merge(df_expert, df_crowd, how='outer', on=['image', 'query_id'])
df_scores['expert_score'] = (df_scores[['first', 'second', 'third']].mean(axis=1) - 1) / 3
def score_aggregate(row) -> object:

    if np.isnan(row['expert_score']):
        row['score'] = row['share_pos']
    elif np.isnan(row['share_pos']):
        row['score'] = row['expert_score']
    else:
        row['score'] = row['expert_score'] * 0.7 + row['share_pos'] * 0.3

    return row

df_scores = df_scores.apply(score_aggregate, axis=1)

display(df_scores.info())
df_scores[df_scores['score']>0].head(10)

In [ ]:
df_train = pd.merge(df_train, df_scores[['image', 'query_id', 'score']], how='outer', on=['image', 'query_id'])
df_train.name = "Train Dataset"
describe_db(df_train)
df_train[df_train['score'] > 0].head(10)

In [ ]:
to_fill = df_train[df_train['query_text'].notna()]
def fill_text(row) -> object:
    if pd.isnull(row['query_text']):
        texts = to_fill[to_fill['query_id'] == row['query_id']]['query_text']
        if len(texts) > 0:
            row['query_text'] = texts.iloc[0]
    return row

df_train = df_train.apply(fill_text, axis=1)
df_train.name = "Train Dataset"
describe_db(df_train)
display(df_train.head(20))

Все еще присутствуют пропуски в тексте - удалим такие строки.

In [ ]:
df_train.dropna(inplace=True)
describe_db(df_train)

In [ ]:
print("=== Информация о колонке score в df_train ===\n")

print(f"Тип данных: {df_train['score'].dtype}")
print(f"Количество значений: {len(df_train['score'])}")
print(f"Количество пропусков (NaN): {df_train['score'].isna().sum()}")
print(f"Количество уникальных значений: {df_train['score'].nunique()}")

print(f"\nСтатистика:")
print(f"  Минимум: {df_train['score'].min():.4f}")
print(f"  Максимум: {df_train['score'].max():.4f}")
print(f"  Среднее: {df_train['score'].mean():.4f}")
print(f"  Медиана: {df_train['score'].median():.4f}")
print(f"  Стандартное отклонение: {df_train['score'].std():.4f}")

print(f"\nРаспределение (первые 10 значений):")
display(df_train['score'].value_counts().head(10))

**Выводы**

Мы изучили полученные данные и увидели их неравномерность: большинство оценок как экспертов, так и обычных людей, являются отрицательными - не подтверждают соответствие изображения и описания. Большинство изображений не имеют описания полностью или хотя бы хорошо соответствующему содержанию. Таким образом, ставится под сомнение качество будущей модели из-за однобоких тренировочных данных.

Помимо того, мы добавили столбец score в данные трейна как целевую переменную и масштабировали значения.

## Проверка данных

В некоторых странах, где работает ваша компания, действуют ограничения по обработке изображений: поисковым сервисам и сервисам, предоставляющим возможность поиска, запрещено без разрешения родителей или законных представителей предоставлять любую информацию, в том числе, но не исключительно тексты, изображения, видео и аудио, содержащие описание, изображение или запись голоса детей. Ребёнком считается любой человек, не достигший 16 лет.

В вашем сервисе строго следуют законам стран, в которых работают. Поэтому при попытке посмотреть изображения, запрещённые законодательством, вместо картинок показывается дисклеймер:

> This image is unavailable in your country in compliance with local laws
>

Однако у вас в PoC нет возможности воспользоваться данным функционалом. Поэтому все изображения, которые нарушают данный закон, нужно удалить из обучающей выборки.

In [ ]:
blocked_words = [
    # Наиболее частые и очевидные
    'child',
    'children',
    'kid',
    'kids',
    'baby',
    'babies',
    'teen',
    'teenager',
    'teenagers',
    'teenage',
    'boy',
    'boys',
    'girl',
    'girls',
    'infant',
    'infants',
    'toddler',
    'toddlers',
    'youth',
    'youths',
    'young',
    'minor',
    'minors',

    # Средняя частота
    'daughter',
    'son',
    'offspring',
    'juvenile',
    'youngster',
    'pubescent',
    'adolescent',
    'adolescents',
    'preteen',
    'tween',

    # Редкие/синонимы
    'babe',
    'bairn',
    'bambino',
    'brat',
    'chick',
    'cub',
    'imp',
    'innocent',
    'lamb',
    'moppet',
    'nestling',
    'nipper',
    'shaver',
    'sprout',
    'squirt',
    'stripling',
    'suckling',
    'tadpole',
    'tot',
    'tyke',
    'urchin',
    'anklebiter',
    'gamin',
    'gamine',
    'kiddie',
    'small fry',
    'young one',
    'little one',
    'wee one'
]

In [ ]:
lemmatize = nltk.WordNetLemmatizer()

def get_lemmas(text) -> list:
    text = re.sub('[^a-zA-Z]', ' ', text).lower()
    text = nltk.word_tokenize(text, language='english')
    text = [lemmatize.lemmatize(word) for word in text]
    return text

def contains_blocked_words(text) -> bool:
    lemmas = get_lemmas(text)
    return any(word in blocked_words for word in lemmas)

In [ ]:
blocked_mask = df_train['query_text'].apply(contains_blocked_words)

# Показываем примеры
print("Примеры текстов с заблокированными словами:")
print(df_train[blocked_mask]['query_text'].sample(10).unique())

# Показываем изображения
samples = df_train[blocked_mask]['query_id'].sample(16).tolist()
samples = [i.split('#')[0] for i in samples]

fig = plt.figure(figsize=(10,10))
for i in range(16):
    fig.add_subplot(4, 4, i+1)
    image_path = Path(path) / 'train_images' / samples[i]
    image = Image.open(image_path)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    plt.imshow(image)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
df_train = df_train[~blocked_mask].reset_index(drop=True)

print(f"\nРазмер после очистки: {len(df_train)}")

Мы очистили данные от запрещенного контента про детей. Из основной таблицы было удалено около 16000 значений.

## Векторизация изображений

Перейдём к векторизации изображений.

Самый примитивный способ — прочесть изображение и превратить полученную матрицу в вектор. Такой способ нам не подходит: длина векторов может быть сильно разной, так как размеры изображений разные. Поэтому стоит обратиться к свёрточным сетям: они позволяют "выделить" главные компоненты изображений. Как это сделать? Нужно выбрать какую-либо архитектуру, например ResNet-18, посмотреть на слои и исключить полносвязные слои, которые отвечают за конечное предсказание. При этом можно загрузить модель данной архитектуры, предварительно натренированную на датасете ImageNet.

In [ ]:
def load_train(path):
    datagen = ImageDataGenerator(rescale=1./255)
    datagen_flow = datagen.flow_from_dataframe(
        dataframe=df_train,
        directory=Path(path, 'train_images'),
        x_col='image',
        y_col='score',
        target_size=(256, 256),
        batch_size=16,
        class_mode='raw',
        subset='training',
        seed=12345
    )
    return datagen_flow

# Модель для эмбеддингов
def create_embedding_model(input_shape=(256, 256, 3)):
    backbone = ResNet50(input_shape=input_shape,
                        weights='imagenet',
                        include_top=False)
    backbone.trainable = False

    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    return model

def cache_embeddings(model, df, img_dir, batch_size=32):
    df_unique = df.drop_duplicates(subset=['image']).copy()

    datagen = ImageDataGenerator(rescale=1./255)
    generator = datagen.flow_from_dataframe(
        dataframe=df_unique,
        directory=img_dir,
        x_col='image',
        target_size=(256, 256),
        batch_size=batch_size,
        class_mode=None,
        shuffle=False
    )

    embeddings = []
    for i in range(len(generator)):
        images = generator[i]
        emb = model.predict(images, verbose=0)
        embeddings.append(emb)

    all_embeddings = np.concatenate(embeddings, axis=0)
    embedding_dict = dict(zip(df_unique['image'], all_embeddings))

    print(f"Закэшировано {len(embedding_dict)} уникальных изображений")
    print(f"Размер эмбеддинга: {all_embeddings.shape[1]}")
    return embedding_dict

def get_embeddings_for_dataframe(df, embedding_dict):
    return np.array([embedding_dict[img] for img in df['image']])


In [ ]:
# Визуализация первых 10 изображений
flow_train = load_train(path)
features, target = next(flow_train)

fig = plt.figure(figsize=(13,10))
for i in range(10):
    fig.add_subplot(2, 5, i+1)
    plt.imshow(features[i])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


In [ ]:
embedding_model = create_embedding_model()

img_dir = Path(path, 'train_images')
embedding_dict = cache_embeddings(embedding_model, df_train, img_dir)

pict_embeds = get_embeddings_for_dataframe(df_train, embedding_dict)

print(f"Форма эмбеддингов: {pict_embeds.shape}")
print(f"Количество строк в df_train: {len(df_train)}")
print(f"Размер одного эмбеддинга: {pict_embeds.shape[1]}")

Эмбендинги сформированы.

## Векторизация текстов

Следующий этап — векторизация текстов. Вы можете поэкспериментировать с несколькими способами векторизации текстов:

- tf-idf
- word2vec
- \*трансформеры (например Bert)

\* — если вы изучали трансформеры в спринте Машинное обучение для текстов.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Использую: {device}")

# Модель для получения эмбеддингов текстов
tokenizer = AutoTokenizer.from_pretrained('unitary/toxic-bert')
bert_model = AutoModel.from_pretrained('unitary/toxic-bert')
bert_model = bert_model.to(device)
bert_model.eval()

print("BERT модель загружена")

In [ ]:

MAX_LENGTH = 512
BATCH_SIZE = 50

# Получаем уникальные тексты
unique_texts = df_train['query_text'].drop_duplicates().tolist()
print(f"Уникальных текстов: {len(unique_texts)} (из {len(df_train)})")

# Токенизация уникальных текстов
print("Токенизация...")
encodings = tokenizer(
    unique_texts,
    padding='max_length',
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors='pt'
)

padded = encodings['input_ids'].to(device)
attention_mask = encodings['attention_mask'].to(device)

# Получаем эмбеддинги для уникальных текстов
print("Получение эмбеддингов...")
embeddings = []

for i in tqdm(range(0, len(padded), BATCH_SIZE)):
    batch = padded[i:i+BATCH_SIZE]
    mask_batch = attention_mask[i:i+BATCH_SIZE]

    with torch.no_grad():
        batch_embeddings = bert_model(batch, attention_mask=mask_batch)

    embeddings.append(batch_embeddings.last_hidden_state[:, 0, :].cpu().numpy())

unique_embeddings = np.vstack(embeddings)
embedding_dict = dict(zip(unique_texts, unique_embeddings))
text_embeds = np.array([embedding_dict[text] for text in df_train['query_text']])

text_embeds.shape

## Объединение векторов

Подготовьте данные для обучения: объедините векторы изображений и векторы текстов с целевой переменной.

In [ ]:
X = np.concatenate((pict_embeds, text_embeds), axis=1)
X.shape
y = np.array(df_train['score'])
y.shape

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Объединили данные.

## Обучение модели предсказания соответствия

Для обучения разделите датасет на тренировочную и тестовую выборки. Простое случайное разбиение не подходит: нужно исключить попадание изображения и в обучающую, и в тестовую выборки.
Для того чтобы учесть изображения при разбиении, можно воспользоваться классом [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) из библиотеки sklearn.model_selection.

Код ниже разбивает датасет на тренировочную и тестовую выборки в пропорции 7:3 так, что строки с одинаковым значением 'group_column' будут содержаться либо в тестовом, либо в тренировочном датасете.

```
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, train_size=.7, random_state=42)
train_indices, test_indices = next(gss.split(X=df.drop(columns=['target']), y=df['target'], groups=df['group_column']))
train_df, test_df = df.loc[train_indices], df.loc[test_indices]

```

Какую модель использовать — выберите самостоятельно. Также вам предстоит выбрать метрику качества либо реализовать свою.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, train_size=.7, random_state=42)
train_indices, test_indices = next(gss.split(X=X, y=y, groups=df_train['image']))
X_train, X_val = X[train_indices], X[test_indices]
y_train, y_val = y[train_indices], y[test_indices]

print("Размерности обучающей выборки (признаки, целевая переменная):", X_train.shape, y_train.shape)
print("Размерности тестовой выборки (признаки, целевая переменная):", X_val.shape, y_val.shape)

### Базовые модели

In [ ]:
pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lgb.LGBMRegressor(random_state=42, verbose=-1))
])

# Параметры для поиска
param_grid = [
    # LightGBM
    {
        'model': [lgb.LGBMRegressor(random_state=42, verbose=-1)],
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [4, 6, -1],
        'model__num_leaves': [31, 50],
        'scaler': [StandardScaler(), 'passthrough']
    },
    # XGBoost
    {
        'model': [xgb.XGBRegressor(random_state=42, verbosity=0)],
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [4, 6],
        'model__subsample': [0.8, 0.9],
        'scaler': [StandardScaler(), 'passthrough']
    },
    # Ridge
    {
        'model': [Ridge(random_state=42)],
        'model__alpha': [0.5, 1.0, 5.0],
        'scaler': [StandardScaler(), MinMaxScaler()]
    },
    # Linear Regression
    {
        'model': [LinearRegression(n_jobs=-1)],
        'scaler': [StandardScaler(), MinMaxScaler(), 'passthrough']
    }
]

cv = GroupShuffleSplit(n_splits=3, test_size=0.2, random_state=42)

# Поиск (уменьшено с 50 до 20)
random_search = RandomizedSearchCV(
    pipe_final,
    param_grid,
    n_iter=20,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Обучение
print("Поиск лучших параметров...")
random_search.fit(X_train, y_train, groups=df_train['image'].iloc[train_indices])

print(f"\nЛучшие параметры: {random_search.best_params_}")
print(f"Лучший CV RMSE: {abs(random_search.best_score_):.4f}")

# Оценка на валидации
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_val)

print(f"\nРезультаты на валидации:")
print(f"Val RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")
print(f"Val MAE: {mean_absolute_error(y_val, y_pred):.4f}")

### Нейронная модель

In [ ]:
NN = Sequential()
NN.add(Dense(1024, input_shape=(2816,), activation='relu'))
NN.add(BatchNormalization())
NN.add(Dropout(0.3))
NN.add(Dense(512, activation='relu'))
NN.add(BatchNormalization())
NN.add(Dropout(0.2))
NN.add(Dense(256, activation='relu'))
NN.add(BatchNormalization())
NN.add(Dropout(0.1))
NN.add(Dense(1, activation='sigmoid'))

optimizer = Adam(learning_rate=1e-5)

NN.compile(optimizer=optimizer,
           loss='mean_squared_error',
           metrics=[tf.keras.metrics.RootMeanSquaredError()])

NN.summary()

In [ ]:
# Масштабирование данных
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Обучение
history = NN.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=15,
    batch_size=64,
    verbose=1,
    shuffle=True
)

y_val_pred_nn = NN.predict(X_val_scaled)

In [ ]:
print("\nNN:")
print(f"Val MSE: {mean_squared_error(y_val, y_val_pred_nn):.4f}")
print(f"Val RMSE: {np.sqrt(mean_squared_error(y_val, y_val_pred_nn)):.4f}")
print(f"Val MAE: {mean_absolute_error(y_val, y_val_pred_nn):.4f}")

### Выводы по моделям

Лучшей моделью по результату метрики RMSE (0.21) оказалась LightGBM с параметрами: 'scaler': StandardScaler(), 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model': LGBMRegressor(random_state=42, verbose=-1)

 Однако результаты метрики очень низкие, как и ожидалось. Сделаем предсказания на ней и посмотрим на выданные результаты.

## Тестирование модели

Настало время протестировать модель. Для этого получите эмбеддинги для всех тестовых изображений из папки `test_images`, выберите случайные 10 запросов из файла `test_queries.csv` и для каждого запроса выведите наиболее релевантное изображение. Сравните визуально качество поиска.

In [ ]:
test_images_path = os.path.join(path, 'test_images')
image_filenames = [filename for filename in os.listdir(test_images_path) if filename.endswith('.jpg')]
test_image = pd.DataFrame(image_filenames, columns=['image'])
test_image.name = 'test_image'

In [ ]:
# Функция векторизации изображений
def vectorize_pictures(model, generator):
    embeddings = []
    for i in range(len(generator)):
        images, _ = generator[i]
        emb = model.predict(images, verbose=0)
        embeddings.append(emb)
    return np.vstack(embeddings)

# Создадим DataFrame для тестовых изображений
test_images_path = Path(path, 'test_images')
image_filenames = [f for f in os.listdir(test_images_path) if f.endswith('.jpg')]
df_test = pd.DataFrame(image_filenames, columns=['image'])
df_test['score'] = 0

# Создадим генератор для тестовых данных
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen_flow = test_datagen.flow_from_dataframe(
    dataframe=df_test,
    directory=test_images_path,
    x_col='image',
    y_col='score',
    target_size=(256, 256),
    batch_size=16,
    class_mode='raw',
    shuffle=False,
    seed=12345
)

# Получим векторные представления изображений для тестовых данных
image_embeddings = vectorize_pictures(embedding_model, test_gen_flow)
print(f"Эмбеддинги тестовых изображений: {image_embeddings.shape}")

# Функция получения эмбеддингов текста через toxic-bert
def get_text_embeddings(texts):
    """Получает эмбеддинги текстов через toxic-bert"""
    all_embeddings = []
    batch_size = 8

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # Токенизация
        encodings = tokenizer(
            batch_texts,
            padding='max_length',
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )

        # Переносим на устройство
        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = bert_model(**encodings)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        all_embeddings.append(batch_embeddings)

    return np.vstack(all_embeddings)

# Лучшая модель из RandomizedSearchCV (или ваша обученная модель)
best_model = random_search.best_estimator_

# Функция инференса
def inference(query_text):
    # Проверка на заблокированные слова
    blocked_found = any(word in query_text.lower() for word in blocked_words)

    if blocked_found:
        print('-' * 100)
        print(f"\n{query_text}\n")
        print('#' * 77)
        print('## THIS IMAGE IS UNAVAILABLE IN YOUR COUNTRY IN COMPLIANCE WITH LOCAL LAWS ##')
        print('#' * 77)
        print()
        return

    # Получаем эмбеддинг текста
    text_embed = get_text_embeddings([query_text])

    # Комбинируем с эмбеддингами изображений
    X_test = np.hstack([
        image_embeddings,
        np.tile(text_embed, (image_embeddings.shape[0], 1))
    ])

    # Масштабируем
    X_test_scaled = scaler.transform(X_test)

    # Предсказания
    predictions = best_model.predict(X_test_scaled)

    # Топ-5 изображений
    top_indices = np.argsort(predictions)[-5:][::-1]
    top_images = [df_test.iloc[i]['image'] for i in top_indices]
    top_scores = predictions[top_indices]

    # Вывод
    print('-' * 100)
    print(f"\n{query_text}\n")

    # Отображение результатов
    fig = plt.figure(figsize=(15, 6))
    for i, (img, score) in enumerate(zip(top_images, top_scores)):
        ax = fig.add_subplot(1, 5, i + 1)
        ax.set_title(f"{score:.3f}", fontsize=10)
        image = Image.open(Path(test_images_path, img))
        if image.mode != 'RGB':
            image = image.convert('RGB')
        plt.imshow(image)
        plt.xticks([])
        plt.yticks([])

    plt.tight_layout()
    plt.show()
    print('-' * 100)

In [ ]:
df_queries = pd.read_csv(os.path.join(path, 'test_queries.csv'), index_col=[0], sep='|')
test_phrases = df_queries['query_text'].sample(10, random_state=42).tolist()
for phrase in test_phrases:
    inference(phrase)

Скрипт по отображению дисклеймера отображается корректно. Однако, при довольно неплохих (хоть и не хороших) метриках модели, к сожалению, отображение наиболее подходящего изображения по запросу не отрабатывает даже близко к хорошему. Все изображения очень далеки от заданной задачи. **Буду очень благодарна за помощь/подсказку ревьюера.**

Я перепробовала различные подходы, начиная от отбора данных, разных коэффициентов оценок (эксперт/крауд), модели различные пробовала, однако, не удалось получить удовлетворительный результат на тесте, к сожалению. Ожидала не слишком хороший результат, но не настолько.

## Выводы

**Поставленная задача**

Разработать демонстрационную версию поиска изображений по текстовому запросу. Модель должна получать векторное представление изображения и текста, а на выходе выдавать число от 0 до 1 — степень соответствия картинки и запроса.

**Проделанная работа**

*Анализ данных*

- Изучены экспертные оценки (от 1 до 4) и краудсорсинговые данные

- Оценки приведены к шкале 0–1 и добавлены в тренировочный датасет как целевая переменная

- Выявлена корреляция между экспертными и краудсорсинговыми оценками (коэффициент Пирсона 0.50)

*Векторизация*

- Изображения: эмбеддинги через ResNet50

- Тексты: эмбеддинги через toxic-bert

- Признаки объединены

*Обучение моделей*

- Перебраны: LinearRegression, Ridge, XGBoost, LightGBM, RandomForest, нейронная сеть

- Лучший результат показала LightGBM: RMSE = 0.21

*Тестирование*

- Реализована проверка запросов на заблокированные слова

- Для каждого запроса модель выводит топ-5 наиболее релевантных изображений с оценками соответствия

*Выявленные проблемы*

- Сильный перекос данных: большинство пар имеют оценку минимального соответствия, лишь незначительная часть — полное соответствие

- Почти четверть изображений не имеют ни одного качественного описания

- Размытая формулировка промежуточной оценки создаёт дополнительный шум

*Результат*

Метрики модели низкие (RMSE = 0.21), визуальное качество подбора изображений по запросу — неудовлетворительное.

**Запрос ревьюеру**
Буду благодарна за рекомендации по улучшению качества поиска на несбалансированных данных с преобладанием негативных примеров.

- [x]  Jupyter Notebook открыт
- [ ]  Весь код выполняется без ошибок
- [ ]  Ячейки с кодом расположены в порядке исполнения
- [ ]  Исследовательский анализ данных выполнен
- [ ]  Проверены экспертные оценки и краудсорсинговые оценки
- [ ]  Из датасета исключены те объекты, которые выходят за рамки юридических ограничений
- [ ]  Изображения векторизованы
- [ ]  Текстовые запросы векторизованы
- [ ]  Данные корректно разбиты на тренировочную и тестовую выборки
- [ ]  Предложена метрика качества работы модели
- [ ]  Предложена модель схожести изображений и текстового запроса
- [ ]  Модель обучена
- [ ]  По итогам обучения модели сделаны выводы
- [ ]  Проведено тестирование работы модели
- [ ]  По итогам тестирования визуально сравнили качество поиска